# InsPLAD — Gold v2 Vision Expert Selection (resume-safe)

이 notebook은 Drive에 이미 생성된 결과를 우선 재사용합니다.

- 기존 `leaderboard.csv`에 2 models × 4 experts × 3 evidence = 24개 조합이 모두 있으면 추론을 다시 하지 않습니다.
- 결과가 부족할 때만 기존 smoke artifact를 이용해 필요한 실험을 이어서 수행합니다.
- 요약은 `summarize_gold_v2_vision_expert_selection.py`를 사용해 `accuracy` 컬럼 문제 없이 계산합니다.
- 마지막에 핵심 결과를 GitHub `results/latest/gold_v2_vision_expert_selection/`에 자동 publish합니다.

비교:
- Vision Expert: PatchCore / Dinomaly / AnomalyCLIP / ReConPatch-lite
- Evidence: score / bbox / heatmap
- LVLM: Qwen3.5-4B / InternVL3.5-8B


In [ ]:
# 0) 설정
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"
PREVIOUS_RUN_FOLDER = "vision_expert_smoke_v1"
RUN_FOLDER = "gold_v2_vision_expert_selection"
SMOKE_SIZE = 50
SEED = 42
MODELS_TO_RUN = ["qwen3_5_4b", "internvl3_5_8b"]
EXPERTS_TO_RUN = ["patchcore", "dinomaly", "anomalyclip", "reconpatch_lite"]
CONDITIONS = ["score", "bbox", "heatmap"]
RECONPATCH_TRAIN_STEPS = 120
RECONPATCH_MAX_TRAIN_PATCHES = 6000
RECONPATCH_MEMORY_RATIO = 0.05
RECONPATCH_BATCH_SIZE = 192
CHECKPOINT_EVERY = 1
UNSUP_ZIP_NAME = "unsupervised_anomaly_detection.zip"
EXPECTED_UNSUP_ZIP_SIZE = 1_170_967_719
EXPECTED_COMBINATIONS = len(MODELS_TO_RUN) * len(EXPERTS_TO_RUN) * len(CONDITIONS)


In [ ]:
# 1) Drive + 최신 repo clone + 환경 준비
import base64, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
RESULT_BASE = DRIVE_ROOT / "lvlm_insplad_results"
PREV_ROOT = RESULT_BASE / PREVIOUS_RUN_FOLDER
RUN_ROOT = RESULT_BASE / RUN_FOLDER
EVAL_DIR = RUN_ROOT / "eval"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)
REPO_DIR = Path("/content/lvlm_insplad")
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)

def run(cmd, cwd=None, env=None, check=True):
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:10]) + (" ..." if len(cmd) > 10 else ""))
    return subprocess.run(cmd, cwd=cwd, env=env, check=check)

github_token = userdata.get("GITHUB_TOKEN")
if not github_token: raise RuntimeError("Colab Secret GITHUB_TOKEN이 없습니다.")
os.environ["GITHUB_TOKEN"] = github_token
auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy(); git_env.update({"GIT_CONFIG_COUNT":"1","GIT_CONFIG_KEY_0":"http.extraHeader","GIT_CONFIG_VALUE_0":f"AUTHORIZATION: basic {auth}","GIT_TERMINAL_PROMPT":"0"})
run(["git","clone","--depth","1","--branch",REPO_REF,f"https://github.com/{REPO}.git",REPO_DIR], env=git_env)
del auth, git_env
try: hf_token = userdata.get("HF_TOKEN")
except Exception: hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
run([sys.executable,"-m","pip","install","-q","transformers==5.14.0","accelerate==1.14.0","sentencepiece==0.2.2","pandas>=2.2","numpy>=1.26","pillow>=10","tqdm>=4.66"])
run([sys.executable,"-m","pip","install","-q","-e",f"{REPO_DIR}[dev]"])
print("repo:", subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO_DIR,text=True).strip())
print("run:", RUN_ROOT)
print("eval:", EVAL_DIR)


In [ ]:
# 2) Drive 결과 상태 확인 — 24조합이 이미 있으면 heavy inference는 전부 건너뜀
import pandas as pd

def eval_complete():
    path = EVAL_DIR / "leaderboard.csv"
    if not path.is_file() or path.stat().st_size == 0: return False
    df = pd.read_csv(path)
    required = {"model_key", "expert", "condition"}
    if not required.issubset(df.columns): return False
    selected = df[df["model_key"].isin(MODELS_TO_RUN) & df["expert"].isin(EXPERTS_TO_RUN) & df["condition"].isin(CONDITIONS)].drop_duplicates(["model_key","expert","condition"])
    print(f"기존 조합: {len(selected)}/{EXPECTED_COMBINATIONS}")
    return len(selected) == EXPECTED_COMBINATIONS

HAS_COMPLETE_EVAL = eval_complete()
if HAS_COMPLETE_EVAL:
    print("PASS: Drive에 24개 조합 결과가 모두 있습니다. #3의 재추론은 자동 SKIP됩니다.")
else:
    print("INFO: 결과가 불완전합니다. #3에서 기존 checkpoint를 재사용하며 부족한 부분만 이어서 실행합니다.")


In [ ]:
# 3) 필요한 경우에만 Vision Expert / LVLM 실험 이어서 실행
if HAS_COMPLETE_EVAL:
    print("SKIP heavy inference: 기존 Drive leaderboard 재사용")
else:
    import torch
    required = {"qa": PREV_ROOT / "data/aligned_qa" / f"one_shot_plus_smoke_{SMOKE_SIZE}.jsonl", "workspace": PREV_ROOT / "workspace" / "workspace_manifest.csv", "workspace_config": PREV_ROOT / "workspace" / "workspace_config.json", "base_roi_manifest": PREV_ROOT / "vision_experts" / "roi_manifest.csv"}
    missing = [f"{name}: {path}" for name, path in required.items() if not path.is_file()]
    if missing: raise FileNotFoundError("기존 vision expert smoke artifact가 필요합니다.\n" + "\n".join(missing))
    SMOKE_QA = required["qa"]; WORKSPACE = PREV_ROOT / "workspace"; BASE_ROI_MANIFEST = required["base_roi_manifest"]; BASE_ARTIFACT_ROOT = PREV_ROOT
    def find_unique_file(filename):
        matches = [p for p in DRIVE_ROOT.rglob(filename) if p.is_file()]
        unique, seen = [], set()
        for p in matches:
            key = str(p.resolve())
            if key not in seen: unique.append(p); seen.add(key)
        if not unique: raise FileNotFoundError(filename)
        size_matches = [p for p in unique if p.stat().st_size == EXPECTED_UNSUP_ZIP_SIZE]
        return size_matches[0] if size_matches else unique[0]
    UNSUP_ZIP = find_unique_file(UNSUP_ZIP_NAME)
    if UNSUP_ZIP.stat().st_size != EXPECTED_UNSUP_ZIP_SIZE: raise RuntimeError(f"unsupervised ZIP size mismatch: {UNSUP_ZIP.stat().st_size}")
    if not torch.cuda.is_available(): raise RuntimeError("결과가 불완전해서 추가 추론이 필요합니다. GPU Runtime으로 실행하세요.")
    print("GPU:", torch.cuda.get_device_name(0))
    RECON_ROOT = RUN_ROOT / "reconpatch_lite"; RECON_MANIFEST = RECON_ROOT / "roi_manifest_reconpatch_lite.csv"
    def recon_complete():
        if not RECON_MANIFEST.is_file(): return False
        df = pd.read_csv(RECON_MANIFEST)
        return len(df) == SMOKE_SIZE and set(df["expert"]) == {"reconpatch_lite"}
    if not recon_complete():
        run([sys.executable, REPO_DIR / "scripts/run_reconpatch_lite_smoke.py", "--workspace-root", WORKSPACE, "--out-dir", RECON_ROOT, "--train-steps", str(RECONPATCH_TRAIN_STEPS), "--max-train-patches", str(RECONPATCH_MAX_TRAIN_PATCHES), "--memory-ratio", str(RECONPATCH_MEMORY_RATIO), "--batch-size", str(RECONPATCH_BATCH_SIZE), "--seed", str(SEED)])
    else: print("재사용 ReConPatch-lite:", RECON_MANIFEST)
    EVIDENCE_ROOT = RUN_ROOT / "evidence"; EVIDENCE_MANIFEST = EVIDENCE_ROOT / "evidence_manifest.csv"
    evidence_ok = False
    if EVIDENCE_MANIFEST.is_file():
        ev = pd.read_csv(EVIDENCE_MANIFEST)
        evidence_ok = len(ev) == SMOKE_SIZE * len(EXPERTS_TO_RUN) and set(ev["expert"]) == set(EXPERTS_TO_RUN)
    if not evidence_ok:
        run([sys.executable, REPO_DIR / "scripts/build_evidence_ablation.py", "--qa", SMOKE_QA, "--workspace-root", WORKSPACE, "--base-roi-manifest", BASE_ROI_MANIFEST, "--base-artifact-root", BASE_ARTIFACT_ROOT, "--reconpatch-manifest", RECON_MANIFEST, "--reconpatch-artifact-root", RECON_ROOT, "--out-dir", EVIDENCE_ROOT])
    else: print("재사용 evidence:", EVIDENCE_MANIFEST)
    run([sys.executable, REPO_DIR / "scripts/run_hf_evidence_ablation.py", "--qa", SMOKE_QA, "--evidence-manifest", EVIDENCE_MANIFEST, "--artifact-root", EVIDENCE_ROOT, "--workspace-root", WORKSPACE, "--unsup-zip", UNSUP_ZIP, "--out-dir", EVAL_DIR, "--models", *MODELS_TO_RUN, "--experts", *EXPERTS_TO_RUN, "--conditions", *CONDITIONS, "--checkpoint-every", str(CHECKPOINT_EVERY)])
    if not eval_complete(): raise RuntimeError("Vision Expert evaluation이 24개 조합까지 완성되지 않았습니다.")
    HAS_COMPLETE_EVAL = True


In [ ]:
# 4) 요약 + winner 생성 — accuracy 컬럼이 없어도 안전하게 계산
run([sys.executable, REPO_DIR / "scripts/summarize_gold_v2_vision_expert_selection.py", "--eval-dir", EVAL_DIR, "--models", *MODELS_TO_RUN, "--experts", *EXPERTS_TO_RUN, "--conditions", *CONDITIONS])
summary = pd.read_csv(EVAL_DIR / "screening_summary.csv")
winner = json.loads((EVAL_DIR / "winner.json").read_text(encoding="utf-8"))
display(summary)
print(json.dumps(winner, ensure_ascii=False, indent=2))


In [ ]:
# 5) GitHub 자동 publish — 실패해도 Drive 결과는 보존
publish_files = ["leaderboard.csv","anomaly_by_asset.csv","run_status.json","screening_summary.csv","winner.json","run_metadata.json"]
cmd = [sys.executable, REPO_DIR / "scripts/publish_results_to_github.py", "--out-dir", EVAL_DIR, "--repo", REPO, "--branch", REPO_REF, "--prefix", "gold_v2_vision_expert_selection", "--files", *publish_files]
publish = run(cmd, env=os.environ.copy(), check=False)
if publish.returncode == 0:
    print("완료: results/latest/gold_v2_vision_expert_selection/")
else:
    print("WARN: GitHub publish만 실패했습니다. Drive 결과와 winner는 그대로 보존되어 있습니다.")
    print("다시 이 셀만 실행하면 publish를 재시도할 수 있습니다.")


## 다음 단계

이 notebook이 끝나면 ChatGPT에 `vision expert 결과 나왔어 확인해봐`라고 하면 됩니다.

다음은 선택된 Vision Expert + evidence winner를 `gold_v2_final` 246장 전체에 고정 적용하고, 가장 좋았던 Normal Reference와 결합하는 단계입니다.
